# 01 · 원본 전체 프레임 → 공개 Hugging Face
바로 아래 셀에서 `RUN_FULLFRAME_EXPORT = True`로 바꾸고 실행하십시오. 기존 Secrets `HF_TOKEN`을 그대로 사용합니다. CPU 작업이며 다운로드·전처리·기존 이관 셀을 먼저 실행할 필요가 없습니다.

기존 Drive의 해제된 원본을 읽고 JPEG·숫자 값을 보존합니다. 기존 D 분할·정규화도 유지합니다. 새 공개 저장소는 `VLADE-konkuk/libero-plus-fullframes`입니다. 128구간 중 게시·검증된 구간은 재시작 시 건너뜁니다. 마지막 `fullframes_export_completed` 또는 `fullframes_export_already_complete` 로그와 원격 `COMPLETE`가 완료 기준입니다.

기존 다운로드·전처리 셀과 출력은 아래 접힌 구역에 보존했습니다. 기존 학습 입력은 아직 희소 프레임 버전을 사용합니다.

In [ ]:
# 원본 전체 프레임 → 승인된 새 공개 HF 데이터셋. 기존 원본·비공개 자료는 보존합니다.
RUN_FULLFRAME_EXPORT = False
if RUN_FULLFRAME_EXPORT:
    from pathlib import Path
    import os, sys, subprocess, hashlib, re
    from google.colab import drive, userdata
    PROJECT_ROOT = Path(globals().get('PROJECT_ROOT', '/content/drive/MyDrive/colab/VLADE'))
    def connect_drive(project_root: Path, drive, *, force: bool=False) -> None:
        """Probe actual file I/O; an existing FUSE directory may be disconnected."""
        if not force:
            try:
                with (project_root/'src/vlade/data/fullframes.py').open('rb') as source:
                    source.read(1)
                return
            except OSError as error:
                if error.errno not in (2,107):
                    raise
                force=error.errno==107
        print('Google Drive 연결을 다시 준비합니다.',flush=True)
        drive.mount('/content/drive',force_remount=force)
        with (project_root/'src/vlade/data/fullframes.py').open('rb') as source:
            source.read(1)


    def run_transfer(command: list, env: dict, log_path: Path, drive, project_root: Path, *, max_reconnects: int=2) -> None:
        """Restart only after a confirmed Drive transport failure; HF units remain sealed."""
        import json,re,subprocess
        for attempt in range(max_reconnects+1):
            disconnected=False
            with log_path.open('a',encoding='utf-8') as log:
                with subprocess.Popen(command,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                        text=True,encoding='utf-8',errors='replace',bufsize=1) as process:
                    for line in process.stdout:
                        clean=line.replace(env['HF_TOKEN'],'[REDACTED]') if env.get('HF_TOKEN') else line
                        clean=re.sub(r'hf_[A-Za-z0-9]{12,}','[REDACTED]',clean)
                        print(clean,end='',flush=True);log.write(clean);log.flush()
                        try:
                            event=json.loads(clean)
                        except (ValueError,TypeError):
                            continue
                        if isinstance(event,dict) and event.get('status')=='fullframes_export_failed' and event.get('error_type')=='OSError':
                            message=event.get('message','')
                            disconnected='[Errno 107]' in message or 'Transport endpoint is not connected' in message
                    returncode=process.wait()
            if returncode==0:
                return
            if not disconnected or attempt==max_reconnects:
                raise RuntimeError('내보내기가 중단되었습니다. 위 fullframes_export_failed 로그를 확인하십시오. 같은 셀로 재개할 수 있습니다.')
            print(f'Drive 연결 끊김: 재연결 후 완료 구간부터 재개합니다 ({attempt+1}/{max_reconnects}).',flush=True)
            connect_drive(project_root,drive,force=True)

    connect_drive(PROJECT_ROOT, drive)
    expected_sources = {'src/vlade/data/tfrecord_frames.py': 'afce3b5f73d553736b6fbe92a658abac2cb161cabc6377d8fef244c9e2862b8b', 'src/vlade/data/fullframes.py': '9b115ca2605eabadad9f3d3e84b122b7a42d96c80977bb59757f8d5fc58ef301', 'src/vlade/storage/__init__.py': '19aea017e8d67fcb8b03dc7b21aaa67b9c262c24c219229f37baa89136aeb117', 'src/vlade/storage/assets.py': '20cab3040f1236d7170d1b2fa55e2b14a7a0d59c2eee4515f80c71a4a7f6302f', 'src/vlade/storage/migrate.py': '2d78b2803348e472991b3d020b936597cbf533a801efd5ce901ce4dc6df7ef09', 'src/vlade/storage/drive_checkpoint.py': 'bca4a1988010dc3c1490663e69658a1205d4785538d6e63441849f625a1187ff', 'scripts/data/prepare_fullframes.py': '0dabdcaecda9c3fa1580dda46c9896dd7031d5df383bc0bb731d176101c48152'}
    source_id = hashlib.sha256(repr(expected_sources).encode()).hexdigest()[:16]
    source_root = Path('/content/vlade_fullframes_source_'+source_id)
    for relative, expected in expected_sources.items():
        data = (PROJECT_ROOT/relative).read_bytes()
        if hashlib.sha256(data).hexdigest() != expected:
            raise ValueError('Drive 코드와 노트북 버전이 다릅니다. 새로고침하십시오: '+relative)
        target = source_root/relative
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(data)
    python = Path('/content/vlade_fullframes_env/bin/python')
    if not python.is_file():
        subprocess.run([sys.executable, '-m', 'venv', '--without-pip', str(python.parent.parent)], check=True)
    versions = {'huggingface-hub':'0.36.2','pyarrow':'19.0.0','protobuf':'5.29.3','google-crc32c':'1.7.1','numpy':'2.2.6'}
    check = 'from importlib.metadata import version; assert all(version(k)==v for k,v in '+repr(versions)+'.items())'
    if subprocess.run([str(python), '-c', check], capture_output=True).returncode:
        print('전체 프레임 내보내기 CPU 환경을 준비합니다.', flush=True)
        subprocess.run([sys.executable, '-m', 'pip', '--python', str(python), 'install',
            '--disable-pip-version-check', *[k+'=='+v for k,v in versions.items()]], check=True)
    env = os.environ.copy()
    env.update(HF_TOKEN=userdata.get('HF_TOKEN'), HF_HOME='/content/vlade_fullframes_hf_cache',
        HF_TOKEN_PATH='/content/vlade_hf_token', HF_HUB_DISABLE_PROGRESS_BARS='1', HF_HUB_DISABLE_XET='1',
        PYTHONUNBUFFERED='1', PYTHONDONTWRITEBYTECODE='1', CUDA_VISIBLE_DEVICES='')
    env.pop('VLADE_HF_TLS12', None)
    work_root = Path('/content/vlade_fullframes_export')
    work_root.mkdir(parents=True, exist_ok=True)
    log_path = work_root/'export.log'
    print('전체 프레임 내보내기 로그:', log_path, flush=True)
    print('목표: 14,347 에피소드 / 2,238,036 프레임. 완료 구간은 검증 후 재사용합니다.', flush=True)
    command = [str(python), '-u', str(source_root/'scripts/data/prepare_fullframes.py'),
        '--tfds-root', str(PROJECT_ROOT/'data/processed/libero_plus_tfds'), '--work-root', str(work_root),
        '--repo-id', 'VLADE-konkuk/libero-plus-fullframes', '--reference-revision', '21a81ee2c9cf120019ed18c5d1cd06bf4336c69e', '--public']
    run_transfer(command, env, log_path, drive, PROJECT_ROOT)
else:
    print('전체 프레임 내보내기를 실행하려면 이 셀의 RUN_FULLFRAME_EXPORT를 True로 바꾸십시오.')

# 기존 비공개 자료 이관 · 완료 기록
기존 데이터와 가중치의 비공개 HF 이관 기록입니다. 재실행할 필요가 없습니다.


In [ ]:
# Existing Drive data/weights -> private HF. This runs CPU transfers, never GPU training.
RUN_HF_MIGRATION = False
if RUN_HF_MIGRATION:
    from pathlib import Path
    import os, sys, subprocess, hashlib
    from google.colab import drive, userdata
    PROJECT_ROOT = Path(globals().get('PROJECT_ROOT', '/content/drive/MyDrive/colab/VLADE'))
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
    expected_sources = {'src/vlade/storage/__init__.py': '19aea017e8d67fcb8b03dc7b21aaa67b9c262c24c219229f37baa89136aeb117', 'src/vlade/storage/assets.py': '20cab3040f1236d7170d1b2fa55e2b14a7a0d59c2eee4515f80c71a4a7f6302f', 'src/vlade/storage/migrate.py': '2d78b2803348e472991b3d020b936597cbf533a801efd5ce901ce4dc6df7ef09', 'src/vlade/storage/drive_checkpoint.py': 'bca4a1988010dc3c1490663e69658a1205d4785538d6e63441849f625a1187ff', 'scripts/data/hf_assets.py': 'd2a650f99b81df8ae8e807a764b6239e79a57cc98644216b7677949dec7ae9cf', 'scripts/data/migrate_hf_assets.py': '4f07b200d70d562435f74689afdfadd82498c1ff42ef59303343261a2e032330'}
    for relative, expected in expected_sources.items():
        if hashlib.sha256((PROJECT_ROOT/relative).read_bytes()).hexdigest() != expected:
            raise ValueError('Migration source changed; reopen the matching notebook: '+relative)
    python = Path('/content/vlade_hf_transfer/bin/python')
    if not python.is_file():
        subprocess.run([sys.executable, '-m', 'venv', '--without-pip', str(python.parent.parent)], check=True)
    installed = subprocess.run([str(python), '-c', "import huggingface_hub; assert huggingface_hub.__version__ == '0.36.2'"], capture_output=True)
    if installed.returncode:
        print('HF 이관 환경을 준비합니다.', flush=True)
        subprocess.run([sys.executable, '-m', 'pip', '--python', str(python), 'install',
                        '--disable-pip-version-check', 'huggingface_hub==0.36.2'], check=True)
    env = os.environ.copy()
    env.update(HF_TOKEN=userdata.get('HF_TOKEN'), HF_HOME='/content/vlade_hf_migration_cache',
               HF_TOKEN_PATH='/content/vlade_hf_token', HF_HUB_DISABLE_PROGRESS_BARS='1', PYTHONUNBUFFERED='1')
    import importlib.util
    spec = importlib.util.spec_from_file_location('vlade_drive_checkpoint', PROJECT_ROOT/'src/vlade/storage/drive_checkpoint.py')
    checkpoint_io = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(checkpoint_io)
    checkpoint = checkpoint_io.prepare_colab_checkpoint(PROJECT_ROOT, python.parent.parent/'checkpoint-source',
        '1YF8pC6rg55KIQrLWAVE5lcvdXAvAhHCB', expected_step=1975)
    log_path = python.parent.parent / 'migration.log'
    print('이관 진행 로그:', log_path, flush=True)
    command = [str(python), '-u', str(PROJECT_ROOT/'scripts/data/migrate_hf_assets.py'), '--root', str(PROJECT_ROOT), '--checkpoint', str(checkpoint)]
    with log_path.open('a', encoding='utf-8') as log:
        with subprocess.Popen(command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
            for line in process.stdout:
                clean = line.replace(env['HF_TOKEN'], '[REDACTED]') if env['HF_TOKEN'] else line
                print(clean, end='', flush=True)
                log.write(clean); log.flush()
            returncode = process.wait()
    if returncode:
        raise RuntimeError('이관이 중단되었습니다. 바로 위 migration_failed의 message와 where를 확인하십시오. 로그: '+str(log_path))
else:
    print('Existing-asset migration is off. Use the HF input preparation cells when migration is complete.')

# 01 · LIBERO Plus 데이터·AVA-VLA 가중치 → Google Drive
CPU 런타임에서 시작할 수 있습니다. 저장소의 일반 코드 파일과 폴더를 Drive에 복사하고 공통 프로젝트 루트를 지정하십시오.
`src/scripts/configs/docs/experiments/third_party/ava_vla`와 루트 문서 구조를 유지하십시오. 소스 목록과 해시는 `experiments/manifests/colab_source.json`으로 검사합니다.
실행 순서: Drive 연결 → 환경 설치 → 공식 고정 revision 다운로드/해시 검증 → 분할 ZIP 결합/CRC 검사·해제 → episode 준비.
[데이터 원본](https://huggingface.co/datasets/Sylvest/libero_plus_rlds/tree/fb0c7029b076030d5d57227229e4f7460def1f7c)
은 **75.5 GB 분할 ZIP 한 개**입니다. `.z01/.z02/.zip`을 각각 해제하지 마십시오.
[AVA 초기 T0](https://huggingface.co/LiAuto-DSR/avavla-libero-4in1/tree/090bf68d234ea13bc78476236f82458b25b797b8)
는 backbone·tokenizer·processor·action head·proprio projector·attention generator 전체를 받습니다.
Drive에는 압축 원본+추출본+query 이미지+가중치를 둘 공간이, `/content`에는 결합 ZIP 약 75.5 GB가 필요합니다.
코드가 실제 파일 크기로 여유 공간을 검사합니다. Drive 계정 할당량은 Drive에서 별도로 확인하십시오.
오늘 로컬에서 다운로드/GPU 학습을 실행한 결과가 아닙니다. 각 셀을 실행하면 해당 다운로드·처리가 시작됩니다.

In [ ]:
from pathlib import Path
from google.colab import drive
import hashlib, json, os, shutil, subprocess, sys
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/colab/VLADE')  # 세 노트북에서 동일한 저장 폴더를 지정하십시오.
ROOT.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT = ROOT  # src/scripts/configs/experiments/third_party가 바로 들어 있는 실제 프로젝트 루트
EXPECTED_MANIFEST_SHA256 = '2fe4f6b49144f4ed6a59b084400da150b1b7c72d0acc3653584990fc95040688'
def stage_source(project_root: Path, scratch_root: Path, expected_manifest_sha256: str) -> Path:
    """Verify Drive source files and stage them without copying data or weights."""
    project_root = project_root.resolve()
    manifest_relative = Path("experiments/manifests/colab_source.json")
    manifest_path = project_root / manifest_relative
    if not manifest_path.is_file():
        raise FileNotFoundError("Drive의 VLADE 폴더에 src/scripts/configs/experiments/third_party 일반 파일 구조를 먼저 복사하십시오: " + str(manifest_path))
    manifest_bytes = manifest_path.read_bytes()
    if hashlib.sha256(manifest_bytes).hexdigest() != expected_manifest_sha256:
        raise ValueError("노트북과 colab_source.json 버전이 다릅니다. 같은 버전의 일반 프로젝트 파일을 사용하십시오.")
    manifest = json.loads(manifest_bytes)
    code = scratch_root / ("vlade_colab_" + expected_manifest_sha256[:12])
    code = code.resolve()
    if code.is_relative_to(project_root) or project_root.is_relative_to(code):
        raise ValueError("실행 소스 임시 폴더와 Drive 프로젝트 폴더는 서로 겹칠 수 없습니다.")
    for relative, expected in manifest["files_sha256"].items():
        path = Path(relative)
        if path.is_absolute() or ".." in path.parts or "\\" in relative or ":" in relative:
            raise ValueError("잘못된 소스 상대 경로: " + relative)
        source = project_root / path
        target = code / path
        if not source.resolve().is_relative_to(project_root) or not target.resolve().is_relative_to(code):
            raise ValueError("프로젝트 밖의 소스 경로: " + relative)
        if not source.is_file() or hashlib.sha256(source.read_bytes()).hexdigest() != expected:
            raise ValueError("Drive 소스가 없거나 변경되었습니다: " + relative)
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.is_file() or hashlib.sha256(target.read_bytes()).hexdigest() != expected:
            shutil.copyfile(source, target)
        if hashlib.sha256(target.read_bytes()).hexdigest() != expected:
            raise ValueError("복사된 소스 무결성 불일치: " + relative)
    # Publish the manifest only after every listed source file has been verified.
    staged_manifest = code / manifest_relative
    staged_manifest.parent.mkdir(parents=True, exist_ok=True)
    temporary_manifest = staged_manifest.with_suffix(".json.tmp")
    temporary_manifest.write_bytes(manifest_bytes)
    temporary_manifest.replace(staged_manifest)
    return code

CODE = stage_source(PROJECT_ROOT, Path('/content'), EXPECTED_MANIFEST_SHA256)
for module_name, module in list(sys.modules.items()):
    if module_name == 'vlade' or module_name.startswith('vlade.'):
        filename = getattr(module, '__file__', None)
        if filename and not Path(filename).resolve().is_relative_to(CODE.resolve()):
            raise RuntimeError('다른 버전의 VLADE가 메모리에 있습니다. Colab 런타임을 다시 시작하십시오.')
sys.path.insert(0, str(CODE / 'src'))
print('Drive:', ROOT, '\n실행 소스:', CODE)
ENV = os.environ.copy()
ENV.update(TOKENIZERS_PARALLELISM='false', WANDB_MODE='disabled', TF_CPP_MIN_LOG_LEVEL='2',
           HF_HOME=str(ROOT / 'hf_cache'), HF_TOKEN_PATH='/content/vlade_hf_token')
def run(args, env=None):
    import signal, threading, time
    deadline = float((env or ENV).get('VLADE_DEADLINE_UNIX', '0'))
    if deadline and deadline - time.time() <= 60:
        raise RuntimeError('8시간 예산이 끝났습니다. 새 작업을 시작하지 않습니다.')
    process = subprocess.Popen([str(x) for x in args], cwd=CODE, env=env or ENV,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    timers = []
    if deadline:
        for delay, callback in [(max(0, deadline-time.time()-60), lambda: process.send_signal(signal.SIGINT)),
                                (max(0, deadline-time.time()), process.kill)]:
            timer = threading.Timer(delay, callback)
            timer.daemon = True
            timer.start()
            timers.append(timer)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise RuntimeError('실행에 실패했거나 시간 예산이 끝났습니다. 위 결과를 확인하십시오.')
    finally:
        for timer in timers:
            timer.cancel()
if shutil.which('uv') is None:
    run([sys.executable, '-m', 'pip', 'install', 'uv'])
UV = shutil.which('uv')
assert UV
def environment(name, version, packages):
    directory = Path('/content') / name
    python = directory / 'bin/python'
    if not python.is_file():
        run([UV, 'venv', '--python', version, directory])
    run([UV, 'pip', 'install', '--python', python, *packages])
    return python
def completed_checkpoints(directory):
    return sorted(path for path in directory.glob('step-*')
                  if (path / 'COMPLETE').is_file() and (path / 'trainer_state.pt').is_file())

# HF inputs are cached on the runtime SSD; Drive retains code and experiment records.
ASSET_CACHE = Path('/content/vlade_hf_assets')
ASSET_WORK_ROOT = Path('/content/vlade_asset_work')
RUNS_ROOT = Path('/content/vlade_runs')
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
ENV.update(VLADE_ASSET_CACHE=str(ASSET_CACHE),
           VLADE_ASSET_REGISTRY=str(CODE/'configs/storage/hf_assets.lock.json'),
           HF_HOME='/content/vlade_hf_home', HF_TOKEN_PATH='/content/vlade_hf_token')
os.environ.update({key: ENV[key] for key in ('VLADE_ASSET_CACHE','VLADE_ASSET_REGISTRY')})
_HF_REGISTRY_READY = False

def hf_input(key, python=None, *, fetch=True):
    global _HF_REGISTRY_READY
    from vlade.storage.assets import asset_record, cache_path
    if fetch and not _HF_REGISTRY_READY:
        if python is None:
            raise ValueError('Select the prepared Python environment before fetching HF assets')
        registry = ASSET_CACHE / 'registry.json'
        run([python, 'scripts/data/hf_assets.py', 'refresh-registry', '--output', registry], env=ENV)
        ENV['VLADE_ASSET_REGISTRY'] = str(registry)
        os.environ['VLADE_ASSET_REGISTRY'] = str(registry)
        _HF_REGISTRY_READY = True
    record = asset_record(key, Path(ENV['VLADE_ASSET_REGISTRY']))
    target = cache_path(key, record, ASSET_CACHE)
    if fetch:
        if python is None:
            raise ValueError('Select the prepared CPU/model Python environment before fetching HF assets')
        run([python, 'scripts/data/hf_assets.py', 'fetch', key,
             '--registry', ENV['VLADE_ASSET_REGISTRY'], '--cache', ASSET_CACHE], env=ENV)
    return target

def hf_path(value, python=None, *, fetch=True):
    value = str(value)
    return hf_input(value[len('hfasset:'):].strip('/\\'), python, fetch=fetch) if value.startswith('hfasset:') else Path(value)

In [ ]:
# HF 설정 후 기존 학습 셀을 실행하면 checkpoint 저장 시 자동 전송합니다.
# 저장소는 HF 웹에서 Private로 먼저 만들고 해당 저장소 쓰기 토큰을 Secrets에 등록하십시오.
if 'ENV' not in globals():
    ENV = os.environ.copy()
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
ENV['HF_TOKEN'] = os.environ['HF_TOKEN']
import datetime, uuid
ENABLE_HUB_UPLOAD = globals().get('ENABLE_HUB_UPLOAD', True)
HF_REPO_ID = 'VLADE-konkuk/VLADE-checkpoints'  # 조직 저장소; 이전 런타임 값도 갱신합니다.
HF_RUN_GROUP = globals().get('HF_RUN_GROUP', datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8])  # 재개 시 기존 group 지정
if ENABLE_HUB_UPLOAD:
    if not HF_RUN_GROUP:
        raise ValueError('HF_RUN_GROUP을 지정하십시오. pilot/main은 출력 폴더명으로 구분합니다.')
    os.environ['VLADE_HUB_REPO'] = HF_REPO_ID
    os.environ['VLADE_HUB_RUN_ID'] = HF_RUN_GROUP
    os.environ['HF_TOKEN_PATH'] = '/content/vlade_hf_token'
    # 모델 캐시는 기존 위치를 유지하고 토큰 파일은 런타임 로컬에만 둡니다.
    ENV.update(HF_TOKEN=os.environ['HF_TOKEN'], VLADE_HUB_REPO=HF_REPO_ID,
               VLADE_HUB_RUN_ID=HF_RUN_GROUP, HF_TOKEN_PATH='/content/vlade_hf_token')
    print('Private Hub 업로드 설정:', HF_REPO_ID, HF_RUN_GROUP)
else:
    for key in ('VLADE_HUB_REPO', 'VLADE_HUB_RUN_ID'):
        ENV.pop(key, None)
        os.environ.pop(key, None)
    print('Hub 업로드 꺼짐: HF_RUN_GROUP 설정 후 ENABLE_HUB_UPLOAD=True로 이 셀을 다시 실행하십시오.')

In [ ]:
DATA_PY = environment('vlade_data_py311', '3.11', [
    'numpy==1.26.4', 'pillow==10.4.0', 'huggingface_hub==0.36.2',
    'tensorflow==2.15.0', 'tensorflow-datasets==4.9.3', 'tensorflow-metadata==1.15.0'])
if shutil.which('zip') is None:
    run(['apt-get', 'update'])
    run(['apt-get', 'install', '-y', 'zip'])

In [ ]:
# 인증·조직 저장소 접근 확인만 수행합니다. 모델·데이터 업로드나 학습은 실행하지 않습니다.
if ENABLE_HUB_UPLOAD:
    run([DATA_PY, '-c', "import json, sys\nfrom huggingface_hub import HfApi\ntry:\n    api = HfApi()\n    account = api.whoami()\n    api.create_repo(repo_id=sys.argv[1], repo_type='model', private=True, exist_ok=True)\n    info = api.model_info(sys.argv[1])\n    if info.private is not True:\n        raise ValueError('Private repository required')\n    print(json.dumps({'authenticated_user': account['name'], 'repo_id': sys.argv[1],\n                      'private': info.private, 'status': 'connection_verified'}))\nexcept Exception as error:\n    print('HF connection failed:', type(error).__name__)\n    raise SystemExit(1)", HF_REPO_ID], env=ENV)
else:
    print('먼저 private-hub 셀에서 업로드 설정을 켜십시오.')

## 공식 파일 다운로드·검증
중단 후 같은 셀을 실행하면 Hub 다운로드 캐시를 재사용하고 파일 해시를 다시 검사합니다. 대용량 데이터를 로컬 컴퓨터로 거치지 않고 Drive에 저장합니다.

In [ ]:
run([DATA_PY, 'scripts/data/prepare_colab_assets.py', 'data', '--root', ROOT])
run([DATA_PY, 'scripts/data/prepare_colab_assets.py', 'ava', '--root', ROOT])

In [ ]:
run([DATA_PY, 'scripts/data/prepare_colab_assets.py', 'extract', '--root', ROOT,
     '--scratch', '/content/libero_zip_scratch'])

## 공통 episode 분할·정규화·관측 준비
80/10/10은 동일 콘텐츠 그룹의 seed 42 해시로 배정합니다. 실제 episode 비율은 근사치입니다.
다른 컴퓨터에서 만든 기존 split과 같은 배정이라고 주장하지 않습니다. 기존 manifest를 받으면 비교 후 한 버전으로 고정하십시오.
Train의 **전체 timestep**에서 exact q01/q99를 계산합니다. Validation/test는 통계에 넣지 않습니다.
Query 간격은 8이며 두 카메라 원본을 query 시점에만 보관합니다. 마지막 chunk의 padding은 loss에서 제외합니다.
이 prepared-data 경로는 augmentation 없이 FFT/BC/cache의 입력·정규화를 공유합니다.
중단된 `.incomplete`는 완료 데이터로 읽지 않습니다. 전처리 중단 시에는 새 output 경로로 다시 준비하십시오.

In [ ]:
PREPARED = ROOT / 'data/processed/libero_plus_episode80'
if (PREPARED / 'COMPLETE').is_file():
    print('준비된 데이터 재사용:', PREPARED)
else:
    run([DATA_PY, 'scripts/data/prepare_episode_data.py',
         '--tfds-root', ROOT / 'data/processed/libero_plus_tfds', '--output', PREPARED])
manifest = json.loads((PREPARED / 'manifest.json').read_text())
print(json.dumps({key: manifest[key] for key in ['counts', 'split_policy', 'split_sha256',
                                               'normalization_sha256', 'dataset_revision']}, indent=2))
print('다음: 02 Teacher 노트북을 A100 런타임에서 실행하십시오.')